# Avance 3 - Machine Learning
## Sistema de gestión de flujo de mercadería y reportes - Tiendas Ripley
Este notebook implementa y valida un modelo de Árbol de Decisión para predecir riesgo operativo.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
np.random.seed(42)


## 1. Dataset simulado
Se generan datos representativos del flujo operativo de mercadería: categoría, camión, cantidad, stock, demanda, reposición e incidencias.


In [ ]:
n = 1500
categorias = ['Ropa', 'Electrodoméstico', 'Tecnología', 'Alimentos', 'Muebles']
camiones = ['Camión 01', 'Camión 03', 'Camión 05', 'Camión 06']
rotaciones = ['Baja', 'Media', 'Alta']

df = pd.DataFrame({
    'categoria': np.random.choice(categorias, n),
    'camion': np.random.choice(camiones, n),
    'cantidad_total': np.random.randint(1, 120, n),
    'numero_productos': np.random.randint(1, 8, n),
    'stock_estimado': np.random.randint(5, 180, n),
    'stock_minimo': np.random.randint(10, 60, n),
    'demanda_semanal': np.random.randint(10, 160, n),
    'unidades_transito': np.random.randint(0, 100, n),
    'tiempo_reposicion_dias': np.random.randint(1, 20, n),
    'dias_sin_reposicion': np.random.randint(1, 40, n),
    'tiene_incidencia': np.random.choice([0, 1], n, p=[0.78, 0.22]),
    'rotacion_producto': np.random.choice(rotaciones, n)
})

df['riesgo_operativo'] = np.where(
    (df['stock_estimado'] < df['stock_minimo']) |
    (df['demanda_semanal'] > df['stock_estimado'] + df['unidades_transito']) |
    ((df['cantidad_total'] > 60) & (df['tiempo_reposicion_dias'] > 8)) |
    ((df['rotacion_producto'] == 'Alta') & (df['stock_estimado'] < 45)) |
    ((df['tiene_incidencia'] == 1) & (df['cantidad_total'] > 25)) |
    ((df['dias_sin_reposicion'] > 25) & (df['stock_estimado'] < 70)),
    'Riesgo', 'Normal'
)
df.to_csv('dataset_ml_ripley.csv', index=False)
df.head()


## 2. Análisis exploratorio de datos


In [ ]:
print('Registros y variables:', df.shape)
print('Valores nulos:')
print(df.isnull().sum())
print('Duplicados:', df.duplicated().sum())
display(df.describe())
print(df['riesgo_operativo'].value_counts())


In [ ]:
plt.figure(figsize=(6,4))
df['riesgo_operativo'].value_counts().plot(kind='bar')
plt.title('Distribución del riesgo operativo')
plt.xlabel('Estado')
plt.ylabel('Cantidad')
plt.show()

plt.figure(figsize=(7,4))
plt.hist(df['cantidad_total'], bins=20)
plt.title('Histograma de cantidad total')
plt.xlabel('Cantidad')
plt.ylabel('Frecuencia')
plt.show()

plt.figure(figsize=(7,4))
plt.boxplot(df['tiempo_reposicion_dias'], vert=False)
plt.title('Boxplot del tiempo de reposición')
plt.xlabel('Días')
plt.show()

plt.figure(figsize=(7,4))
plt.scatter(df['stock_estimado'], df['demanda_semanal'])
plt.title('Stock estimado vs demanda semanal')
plt.xlabel('Stock estimado')
plt.ylabel('Demanda semanal')
plt.show()


In [ ]:
corr = df.select_dtypes(include=['int64','float64']).corr()
display(corr)
plt.figure(figsize=(8,6))
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Matriz de correlación')
plt.show()


## 3. Preparación de datos y entrenamiento


In [ ]:
df_modelo = df.copy()
df_modelo['riesgo_operativo'] = df_modelo['riesgo_operativo'].map({'Normal':0, 'Riesgo':1})
X = df_modelo.drop(columns=['riesgo_operativo'])
y = df_modelo['riesgo_operativo']
X = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
modelo = DecisionTreeClassifier(criterion='gini', max_depth=6, min_samples_split=20, min_samples_leaf=8, random_state=42)
modelo.fit(X_train, y_train)
print('Modelo entrenado correctamente')


## 4. Evaluación del modelo


In [ ]:
y_pred = modelo.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, y_pred),4))
print('Precision:', round(precision_score(y_test, y_pred),4))
print('Recall:', round(recall_score(y_test, y_pred),4))
print('F1 Score:', round(f1_score(y_test, y_pred),4))
print(classification_report(y_test, y_pred, target_names=['Normal','Riesgo']))
matriz = confusion_matrix(y_test, y_pred)
print(matriz)
plt.figure(figsize=(5,4))
plt.imshow(matriz)
plt.colorbar()
plt.xticks([0,1], ['Normal','Riesgo'])
plt.yticks([0,1], ['Normal','Riesgo'])
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.title('Matriz de confusión')
for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        plt.text(j, i, matriz[i,j], ha='center', va='center')
plt.show()


In [ ]:
importancia = pd.DataFrame({'Variable': X.columns, 'Importancia': modelo.feature_importances_}).sort_values(by='Importancia', ascending=False)
display(importancia.head(10))
plt.figure(figsize=(9,5))
top = importancia.head(10).sort_values(by='Importancia')
plt.barh(top['Variable'], top['Importancia'])
plt.title('Variables más importantes')
plt.show()


## 5. Predicción de un nuevo caso


In [ ]:
nuevo = pd.DataFrame([{
    'categoria': 'Electrodoméstico',
    'camion': 'Camión 06',
    'cantidad_total': 24,
    'numero_productos': 1,
    'stock_estimado': 18,
    'stock_minimo': 35,
    'demanda_semanal': 80,
    'unidades_transito': 10,
    'tiempo_reposicion_dias': 9,
    'dias_sin_reposicion': 18,
    'tiene_incidencia': 1,
    'rotacion_producto': 'Alta'
}])
nuevo_codificado = pd.get_dummies(nuevo, drop_first=True).reindex(columns=X.columns, fill_value=0)
pred = modelo.predict(nuevo_codificado)[0]
proba = modelo.predict_proba(nuevo_codificado)[0]
print('Resultado:', 'Riesgo operativo' if pred == 1 else 'Normal')
print('Probabilidad Normal:', round(proba[0]*100, 2), '%')
print('Probabilidad Riesgo:', round(proba[1]*100, 2), '%')
